# A Simple Calculator

This file shows the implementation of a *symbolic calculator* using `Larkita`.
The grammar for the language implemented by this parser is as follows:
$$
\begin{array}{lcl}
  \texttt{stmnt}   & \rightarrow & \;\texttt{IDENTIFIER}\; \texttt{':='} \; \;\texttt{expr}\; \texttt{';'} \\
                   & \mid        & \;\texttt{expr}\; \texttt{';'} \\[0.2cm]
  \texttt{expr}    & \rightarrow & \;\texttt{expr}\; \texttt{'+'} \; \texttt{product}  \\
                   & \mid        & \;\texttt{expr}\; \texttt{'-'} \; \texttt{product}  \\
                   & \mid        & \;\texttt{product}                                  \\[0.2cm]
  \texttt{product} & \rightarrow & \;\texttt{product}\; \texttt{'*'} \;\texttt{factor} \\
                   & \mid        & \;\texttt{product}\; \texttt{'/'} \;\texttt{factor} \\
                   & \mid        & \;\texttt{factor}                                   \\[0.2cm]
  \texttt{factor}  & \rightarrow &   \texttt{'('} \; \texttt{expr} \;\texttt{')'}      \\
                   & \mid        & \;\texttt{NUMBER}                                   \\
                   & \mid        & \;\texttt{IDENTIFIER}
\end{array}
$$

## Setup and Imports
We load the OCaml port of `Larkita`, which is stored in the directory `../lib`.  It generates a *canonical LR(1) parser* that returns the abstract syntax tree of its input as a *nested tuple*.  

In [ ]:
#use "../lib/nb.ml";;
load_lib "larkita.ml"

`Larkita` returns the abstract syntax tree as a *nested tuple* whose first component is the *head*.  Nested
tuples are values of the type `nested`, which is defined in the file `../lib/nested.ml`:
```ocaml
type value =
  | Int   of int
  | Float of float
  | Str   of string
  | Bool  of bool

type nested =
  | Leaf of value
  | Node of nested list
```
A string like `"+"` is the leaf `Leaf (Str "+")`, and a tuple is a node `Node [...]` that holds the list of its
components.  A head consisting of a single literal like `"+"` is a string, and a head consisting of several
literals is the tuple of these literals.  The notebook prints nested tuples in tuple notation:

| Tuple                                      | OCaml value                                                                 |
|--------------------------------------------|-----------------------------------------------------------------------------|
| `("+", l, r)`                              | `Node [Leaf (Str "+"); l; r]`                                               |
| `((":=", ";"), ("IDENTIFIER", x), e)`      | `Node [Node [Leaf (Str ":="); Leaf (Str ";")]; Node [Leaf (Str "IDENTIFIER"); Leaf (Str x)]; e]` |
| `("NUMBER", "42")`                         | `Node [Leaf (Str "NUMBER"); Leaf (Str "42")]`                               |
| `("program",)`                             | `Node [Leaf (Str "program")]`                                               |
| `"-"` (as a child)                         | `Leaf (Str "-")`                                                            |

## Specification of the Grammar

`Larkita` reads grammars in the *extended Backus-Naur Form* (EBNF) of `Lark`:  both the lexical tokens and the
grammar rules are defined in a single grammar.  It is stored in the file `calc.lark`, which the cell below displays.
Let us break down the components of our grammar definition:

* `stmnt`, `expr`, `product`, `factor`: These variables define the hierarchical structure of our mathematical
   expressions, natively enforcing operator precedence (e.g., multiplication in `product` happens before addition
   in `expr`).  The first rule of the grammar defines the *start symbol*, hence a `start` rule is not needed.
* **The abstract syntax tree** is determined by the grammar itself:  the string literals of an alternative form
   the *head* of the tuple that is returned, the remaining symbols become its *children*.
   * `expr "+" product` yields `("+", ast(expr), ast(product))`.  Here `ast(expr)` is the *abstract syntax tree* produced when parsing `expr`.
   * `IDENTIFIER ":=" expr ";"` yields `((":=", ";"), ("IDENTIFIER", name), ast(expr))`:  If there is more than
     one literal, the head is the tuple of all literals.
   * `expr ";"` yields `(";", ast(expr))`.
   * A named token like `NUMBER` yields a pair like `("NUMBER", "42")`.
   * An alternative that consists of a single symbol, like `expr: product`, passes the abstract syntax tree of this
     symbol through.  This is what the `?` prefix achieves in `Lark`.
   As OCaml values, `("+", ast(expr), ast(product))` is `Node [Leaf (Str "+"); ast(expr); ast(product)]`, and
   `("NUMBER", "42")` is `Node [Leaf (Str "NUMBER"); Leaf (Str "42")]`.
* `%ignore "(", ")"` at the end of an alternative removes these literals from the head.  Then the alternative
  `"(" expr ")"` consists of a single symbol and returns the abstract syntax tree of `expr`.
* **Lexical Imports** (`%import common...`): Instead of manually writing regular expressions for common tokens,
   we import standard terminal definitions.  Several imports are separated by commas.
   * `CNAME` parses standard variable names (letters, numbers, underscores) and is aliased to `IDENTIFIER`.
   * `NUMBER` handles both integers and floating-point numbers.
   * `WS` handles whitespace.
* `%ignore WS`: This directs the lexer to completely discard spaces and tabs so they do not clutter our parsing logic.

The next cell displays the grammar that is stored in `calc.lark`.

In [ ]:
cat "calc.lark"

We generate the parser.  `Larkita` always generates a canonical LR(1) parser.  Furthermore, it writes a description
of all LR(1) states into the file specified by `~out_file`.  We will look at this file at the end of this notebook.

In [ ]:
let parser = Larkita.open_ ~out_file:"calc.out" "calc.lark"

# The Abstract Syntax Tree

The parser returns the abstract syntax tree as a nested tuple, so there is no need to convert a parse tree.
For example, the string `x := y + z;` is parsed into the tuple
`((":=", ";"), ("IDENTIFIER", "x"), ("+", ("IDENTIFIER", "y"), ("IDENTIFIER", "z")))`.

In [ ]:
Larkita.parse parser "x := y + z;"

In [ ]:
Larkita.parse parser "(1 + 2) * x;"

## Evaluating the Abstract Syntax Tree

Now we can write our own recursive evaluation function that works directly on the nested tuples returned by the
parser.  The variable `names2Values` stores the values associated with the variables we define.  It is a
reference to an (immutable) dictionary of type `Dict.t`, which is replaced by an extended dictionary on every
assignment.

In [ ]:
let names2Values : (string, float) Dict.t ref = ref Dict.empty

An expression has a value, while a statement has none.  Since a function has a single result type, there are two
functions:  `evaluate_expr` computes the value of an expression, while `evaluate_adt` executes a statement.  Both
match the tuple with *patterns*:  for example, the tuple `((":=", ";"), ("IDENTIFIER", var_name), expr)` is matched by
the pattern `Node [Node [Leaf (Str ":="); Leaf (Str ";")]; Node [Leaf (Str "IDENTIFIER"); Leaf (Str var_name)]; expr]`.
The type `nested` does not restrict the shape of a tuple, so both functions need a final case that raises an error
if they encounter an unknown structure.

In [ ]:
let rec evaluate_expr (adt : nested) : float =
  match adt with
  | Node [ Leaf (Str "NUMBER"); Leaf (Str value) ] -> float_of_string value
  | Node [ Leaf (Str "IDENTIFIER"); Leaf (Str var_name) ] -> (
      match Dict.find_opt var_name !names2Values with
      | Some value -> value
      | None ->
          Printf.printf "Error: variable '%s' is undefined.\n%!" var_name;
          0.0)
  | Node [ Leaf (Str (("+" | "-" | "*" | "/") as op)); left; right ] -> (
      let val_left  = evaluate_expr left  in
      let val_right = evaluate_expr right in
      match op with
      | "+" -> val_left +. val_right
      | "-" -> val_left -. val_right
      | "*" -> val_left *. val_right
      | _   -> val_left /. val_right)
  | _ -> invalid_arg ("Unknown ADT structure: " ^ Nested.to_string adt)

let evaluate_adt (adt : nested) : unit =
  match adt with
  | Node [ Node [ Leaf (Str ":="); Leaf (Str ";") ]; Node [ Leaf (Str "IDENTIFIER"); Leaf (Str var_name) ]; expr ] ->
      names2Values := Dict.add var_name (evaluate_expr expr) !names2Values
  | Node [ Leaf (Str ";"); expr ] -> print_endline (float_repr (evaluate_expr expr))
  | _ -> invalid_arg ("Unknown ADT structure: " ^ Nested.to_string adt)

## Execution

The REPL (Read-Eval-Print Loop) takes the input string, parses it into a nested tuple, and then evaluates it.

In [ ]:
let main () : unit =
  let rec loop () =
    match input "calc> " with
    | "" -> ()
    | s ->
        (try
           (* 1. Parse the string into a nested tuple *)
           let adt = Larkita.parse parser s in
           (* 2. Evaluate the nested tuple *)
           evaluate_adt adt
         with
         | Larkita.Unexpected_input e -> Printf.printf "Error: %s\n%!" e.message
         | e -> Printf.printf "Error: %s\n%!" (Printexc.to_string e));
        loop ()
  in
  loop ()

The notebook cannot read from the keyboard when it is executed without a browser.  Therefore, `simulate_input`
provides the lines that `input` returns.  An empty line ends the loop.

In [ ]:
simulate_input [ "x := 1 + 2 * 3;"; "y := x / 2;"; "x * y - 4;"; "z + 1;"; "(1 + 2;"; "" ];;
main ()

Check the populated memory map:

In [ ]:
!names2Values

We can even get the parser states:  `Larkita` has written them into the file `calc.out`.  This file lists

1. the rules of the grammar, each followed by the abstract syntax tree that a reduction with this rule produces,
2. the conflicts (there are none), and
3. all states of the canonical LR(1) automaton.  Every state lists its marked rules together with their
   *lookahead* tokens, followed by the actions and the gotos of the state.  The marked rules that are added
   by the closure are marked with a `+`.

In [ ]:
cat "calc.out"